In [1]:
# Cell 1
%matplotlib inline
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import pyogrio
from pathlib import Path
from IPython.display import Image as IPImage
from scripts.shared import db_utils

conn = db_utils.db_connect()
ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'cdop'
OUT.mkdir(parents=True, exist_ok=True)

WORLD = gpd.read_file(Path(pyogrio.__file__).parent / 'tests/fixtures/naturalearth_lowres/naturalearth_lowres.shp')

print("Setup complete.")

Setup complete.


In [2]:
# Cell 2 — Load all D-PLACE 'society' rows (7 datasets; excludes 4,085 languoid scaffold rows)
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

sql = """
SELECT id, name, contribution_id, glottocode, xd_id,
       latitude AS lat, longitude AS lon, main_focal_year
FROM dplace.societies
WHERE type = 'society'
ORDER BY contribution_id, id
"""
soc = pd.read_sql(sql, conn)
print(f"Loaded {len(soc):,} society rows across {soc['contribution_id'].nunique()} datasets\n")
print(soc.groupby('contribution_id').size().sort_values(ascending=False).to_string())

Loaded 2,599 society rows across 7 datasets

contribution_id
dplace-dataset-ea           1291
dplace-dataset-ccmc          410
dplace-dataset-binford       339
dplace-dataset-sccs          186
dplace-dataset-wnai          172
dplace-dataset-carneiro4     127
dplace-dataset-carneiro6      74


## Part (a) — Overlap across datasets

`xd_id` (D-PLACE's own cross-dataset identifier) is 100% populated for EA, Binford, SCCS, and
WNAI, and 0% populated for ccmc, carneiro4, and carneiro6 — those three were never cross-linked
by D-PLACE at all. `glottocode` is ~100% populated across all seven datasets and is D-PLACE's
documented fallback for cross-dataset identification when `xd_id` is absent.

**Caveat before reading the next cells:** a shared glottocode means "same language community,"
not necessarily "same culture" — a single language can have many culturally distinct societies
recorded under it. Collisions below are candidates for manual review, not automatic dedup.

In [3]:
# Cell 4 — Part (a): glottocode collisions for the three xd_id-unlinked datasets
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

linked_datasets   = ['dplace-dataset-ea', 'dplace-dataset-binford', 'dplace-dataset-sccs', 'dplace-dataset-wnai']
unlinked_datasets = ['dplace-dataset-ccmc', 'dplace-dataset-carneiro4', 'dplace-dataset-carneiro6']

linked_glotto = set(soc.loc[soc['contribution_id'].isin(linked_datasets), 'glottocode'].dropna())
ea_glotto     = set(soc.loc[soc['contribution_id'] == 'dplace-dataset-ea', 'glottocode'].dropna())

summary = []
for ds in unlinked_datasets:
    g = soc.loc[soc['contribution_id'] == ds, 'glottocode'].dropna()
    summary.append({
        'dataset': ds,
        'n_societies': len(g),
        'glottocode_overlap_vs_any_linked': g.isin(linked_glotto).sum(),
        'glottocode_overlap_vs_ea': g.isin(ea_glotto).sum(),
    })
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

print("\nOverlap among the three unlinked datasets (pairwise):")
for i, ds1 in enumerate(unlinked_datasets):
    g1 = set(soc.loc[soc['contribution_id'] == ds1, 'glottocode'].dropna())
    for ds2 in unlinked_datasets[i+1:]:
        g2 = set(soc.loc[soc['contribution_id'] == ds2, 'glottocode'].dropna())
        print(f"  {ds1} x {ds2}: {len(g1 & g2)} shared glottocodes")

                 dataset  n_societies  glottocode_overlap_vs_any_linked  glottocode_overlap_vs_ea
     dplace-dataset-ccmc          410                               183                       177
dplace-dataset-carneiro4          126                               107                       105
dplace-dataset-carneiro6           74                                64                        61

Overlap among the three unlinked datasets (pairwise):
  dplace-dataset-ccmc x dplace-dataset-carneiro4: 29 shared glottocodes
  dplace-dataset-ccmc x dplace-dataset-carneiro6: 20 shared glottocodes
  dplace-dataset-carneiro4 x dplace-dataset-carneiro6: 49 shared glottocodes


In [4]:
# Cell 5 — Part (a): sample of actual glottocode collisions (ccmc/carneiro vs EA) for eyeballing
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

ea_by_glotto = soc[soc['contribution_id'] == 'dplace-dataset-ea'][['id', 'name', 'glottocode']].rename(
    columns={'id': 'ea_id', 'name': 'ea_name'})

rows = []
for ds in unlinked_datasets:
    other = soc[soc['contribution_id'] == ds][['id', 'name', 'glottocode']].rename(
        columns={'id': 'other_id', 'name': 'other_name'})
    merged = other.merge(ea_by_glotto, on='glottocode', how='inner')
    merged['dataset'] = ds
    rows.append(merged)

collisions = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
print(f"{len(collisions)} rows collide with an EA glottocode\n")
print(collisions[['dataset', 'other_id', 'other_name', 'ea_id', 'ea_name', 'glottocode']].head(30).to_string(index=False))

361 rows collide with an EA glottocode

            dataset     other_id              other_name ea_id       ea_name glottocode
dplace-dataset-ccmc CCMCabkh1244                  Abkhaz  Ci12        Abkhaz   abkh1244
dplace-dataset-ccmc CCMCafar1241                    Afar   Ca6          Afar   afar1241
dplace-dataset-ccmc CCMCainu1240           Hokkaido Ainu   Ec7          Ainu   ainu1240
dplace-dataset-ccmc CCMCalur1250                    Alur  Aj17          Alur   alur1250
dplace-dataset-ccmc CCMCamha1245                 Amharic   Ca7        Amhara   amha1245
dplace-dataset-ccmc CCMCamis1246                    Amis   Ia9           Ami   amis1246
dplace-dataset-ccmc CCMCanga1288             Angami Naga  Ei13        Angami   anga1288
dplace-dataset-ccmc CCMCatay1247                  Atayal   Ia1        Atayal   atay1247
dplace-dataset-ccmc CCMCbafu1246                   Bafut   Ae9           Fut   bafu1246
dplace-dataset-ccmc CCMCbali1278                Balinese   Ib3      Balinese   b

In [10]:
# Cell 6 — Part (a): xd_id overlap for the three already-linked datasets (Binford/SCCS/WNAI) vs EA
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

xd_linked_others = ['dplace-dataset-binford', 'dplace-dataset-sccs', 'dplace-dataset-wnai']
ea_xd = set(soc.loc[soc['contribution_id'] == 'dplace-dataset-ea', 'xd_id'].dropna())

summary2 = []
for ds in xd_linked_others:
    x = soc.loc[soc['contribution_id'] == ds, 'xd_id'].dropna()
    summary2.append({
        'dataset': ds,
        'n_societies': len(x),
        'xd_id_overlap_vs_ea': x.isin(ea_xd).sum(),
        'pct_overlap_vs_ea': round(100 * x.isin(ea_xd).sum() / len(x), 1),
    })
summary2_df = pd.DataFrame(summary2)
print(summary2_df.to_string(index=False))

print("\nPairwise overlap among Binford/SCCS/WNAI themselves:")
for i, ds1 in enumerate(xd_linked_others):
    x1 = set(soc.loc[soc['contribution_id'] == ds1, 'xd_id'].dropna())
    for ds2 in xd_linked_others[i+1:]:
        x2 = set(soc.loc[soc['contribution_id'] == ds2, 'xd_id'].dropna())
        print(f"  {ds1} x {ds2}: {len(x1 & x2)} shared xd_id")

# The number that matters for the "enrich EA, don't pool" framing: how many of EA's 1,291
# societies have a cross-link (via xd_id for Binford/SCCS/WNAI, via glottocode for
# ccmc/carneiro4/carneiro6) into at least one other dataset at all?
other_xd = set(soc.loc[soc['contribution_id'].isin(xd_linked_others), 'xd_id'].dropna())
other_glotto = set(soc.loc[
    soc['contribution_id'].isin(['dplace-dataset-ccmc', 'dplace-dataset-carneiro4', 'dplace-dataset-carneiro6']),
    'glottocode'].dropna())

ea_rows = soc[soc['contribution_id'] == 'dplace-dataset-ea']
ea_has_xd_match = ea_rows['xd_id'].isin(other_xd)
ea_has_glotto_match = ea_rows['glottocode'].isin(other_glotto)
ea_enrichable = ea_has_xd_match | ea_has_glotto_match

print(f"\nEA societies with a potential enrichment match in at least one other dataset: "
      f"{ea_enrichable.sum()} / {len(ea_rows)} ({100 * ea_enrichable.sum() / len(ea_rows):.1f}%)")
print(f"  via xd_id (Binford/SCCS/WNAI): {ea_has_xd_match.sum()}")
print(f"  via glottocode (ccmc/carneiro4/carneiro6): {ea_has_glotto_match.sum()}")

               dataset  n_societies  xd_id_overlap_vs_ea  pct_overlap_vs_ea
dplace-dataset-binford          339                  224               66.1
   dplace-dataset-sccs          186                  186              100.0
   dplace-dataset-wnai          172                  146               84.9

Pairwise overlap among Binford/SCCS/WNAI themselves:
  dplace-dataset-binford x dplace-dataset-sccs: 38 shared xd_id
  dplace-dataset-binford x dplace-dataset-wnai: 103 shared xd_id
  dplace-dataset-sccs x dplace-dataset-wnai: 13 shared xd_id

EA societies with a potential enrichment match in at least one other dataset: 540 / 1291 (41.8%)
  via xd_id (Binford/SCCS/WNAI): 395
  via glottocode (ccmc/carneiro4/carneiro6): 281


## Part (b) — Are the seven datasets' variables commensurable?

Each dataset defines its own variable-ID namespace in D-PLACE (e.g. EA's `EA001`–`EA202`-style
codes are Murdock's own coding scheme). The question is whether datasets that both claim to
measure something like "subsistence" or "settlement pattern" are actually comparable, or just
share a coarse `category` label while coding the underlying construct differently.

In [11]:
# Cell 8 — Part (b): variable category coverage per dataset
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

var_sql = """
SELECT id, name, category, type, unit, contribution_id
FROM dplace.variables
WHERE contribution_id LIKE 'dplace-dataset-%'
"""
varz = pd.read_sql(var_sql, conn)
print(f"{len(varz):,} variables across {varz['contribution_id'].nunique()} dataset contributions\n")

# Diagnostic: pd.crosstab silently drops NaN category rows. Check ccmc directly first
# rather than trust the crosstab's silence.
ccmc_varz = varz[varz['contribution_id'] == 'dplace-dataset-ccmc']
print(f"dplace-dataset-ccmc: {len(ccmc_varz)} variables, "
      f"{ccmc_varz['category'].isna().sum()} with null category")
print("Sample ccmc variable names + categories:")
print(ccmc_varz[['id', 'name', 'category']].head(10).to_string(index=False))
print()

# category is comma-separated in some rows (CLDF separator ', '); take first listed category.
# Fill null category explicitly so it survives the crosstab instead of vanishing.
varz['category_first'] = varz['category'].str.split(',').str[0].str.strip()
varz['category_first'] = varz['category_first'].fillna('(none)')

crosstab = pd.crosstab(varz['contribution_id'], varz['category_first'], dropna=False)
print(crosstab.to_string())

3,341 variables across 14 dataset contributions

dplace-dataset-ccmc: 1 variables, 1 with null category
Sample ccmc variable names + categories:
   id      name category
CCMC1 NHS2 song     None

category_first             (none)  Anthropometry  Architecture  Ceramics and Art  Ceremony  Childhood  Class  Climate  Clothing  Community  Community organization  Data Quality  Death  Demography  Dwelling  Dwellings  Ecology  Economics  Economy  Games  Gender  Gossip  Health  Kinship  Law and Judicial Process  Leadership  Life cycle  Marriage  Material culture  Metalworking  Physical Landscape  Political Organization  Politics  Population  Religion  Ritual  Settlements  Social Organization and Stratification  Special Knowledge and Practices  Subsistence  Tools and Utensils and Textiles  Warfare  Watercraft and Navigation
contribution_id                                                                                                                                                               

In [12]:
# Cell 9 — Part (b): Subsistence-category variables, side by side across datasets
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

subsist = varz[varz['category_first'].str.contains('Subsist', case=False, na=False)]
print(f"{len(subsist)} subsistence-category variables\n")
print(subsist[['contribution_id', 'id', 'name', 'type', 'unit']]
      .sort_values(['contribution_id', 'id']).to_string(index=False))

69 subsistence-category variables

         contribution_id            id                                                                                      name        type unit
dplace-dataset-carneiro4 CARNEIRO4_001                                                                       Agriculture present Categorical None
dplace-dataset-carneiro4 CARNEIRO4_002                                                        Agriculture is primary subsistence Categorical None
dplace-dataset-carneiro4 CARNEIRO4_003                                                 Agricultural plots permanently cultivated Categorical None
dplace-dataset-carneiro4 CARNEIRO4_004                                                           Improved techniques of planting Categorical None
dplace-dataset-carneiro4 CARNEIRO4_005                                                          Soil-working or harvesting tools Categorical None
dplace-dataset-carneiro4 CARNEIRO4_006                                                   

## Part (c) — Spatial distribution

Every society row already carries `lat`/`lon` directly — no basin join needed. Small multiples
per dataset, then all seven overlaid.

In [ ]:
# Cell 11 — Part (c): one map per dataset
print("drawing per-dataset spatial small multiples...")

datasets = sorted(soc['contribution_id'].unique())
colors = plt.cm.tab10(np.linspace(0, 1, len(datasets)))
color_map = dict(zip(datasets, colors))

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.patch.set_facecolor('white')
for ax, ds in zip(axes.flat, datasets):
    WORLD.plot(ax=ax, color='#e8e8e8', edgecolor='#bbbbbb', linewidth=0.4)
    sub = soc[soc['contribution_id'] == ds]
    ax.scatter(sub['lon'], sub['lat'], s=6, color=color_map[ds], alpha=0.8)
    ax.set_title(f"{ds}  (n={len(sub)})", fontsize=10, color='black')
    ax.set_facecolor('white')
    ax.set_xlim(-180, 180); ax.set_ylim(-60, 85)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[len(datasets):]:
    ax.axis('off')
fig.tight_layout()
out_path = OUT / 'dplace_eda_spatial_small_multiples.png'
fig.savefig(out_path, facecolor='white', dpi=130, bbox_inches='tight')
plt.close(fig)
display(IPImage(str(out_path)))

In [ ]:
# Cell 12 — Part (c): all seven datasets overlaid
print("drawing overlay of all datasets...")

fig, ax = plt.subplots(figsize=(16, 9))
fig.patch.set_facecolor('white')
WORLD.plot(ax=ax, color='#e8e8e8', edgecolor='#bbbbbb', linewidth=0.4)
for ds in datasets:
    sub = soc[soc['contribution_id'] == ds]
    ax.scatter(sub['lon'], sub['lat'], s=8, color=color_map[ds], alpha=0.75, label=f"{ds} (n={len(sub)})")
ax.set_facecolor('white')
ax.set_xlim(-180, 180); ax.set_ylim(-60, 85)
ax.set_xticks([]); ax.set_yticks([])
leg = ax.legend(fontsize=8, loc='lower left', framealpha=0.9)
for text in leg.get_texts():
    text.set_color('black')
ax.set_title("D-PLACE society locations — all seven datasets", fontsize=13, color='black', fontweight='bold')
fig.tight_layout()
out_path = OUT / 'dplace_eda_spatial_overlay.png'
fig.savefig(out_path, facecolor='white', dpi=130, bbox_inches='tight')
plt.close(fig)
display(IPImage(str(out_path)))